# 实例分割标签准备 + 训练 (B v2) — AI Studio Notebook 版

把现有 YOLO **bbox 检测**标签转换为**实例分割(mask)** 标签（以 bbox 四角为矩形多边形），
自动组装数据集（复制图片 + 切分 train/val），然后 bootstrap 训练 `yolov8-seg`，
产出的 seg 权重给 App 端 **MetricSizer** 用精确 mask 采样 LiDAR 深度。

> **目录约定**：AI Studio 中**只有 `/home/aistudio/work/` 会被持久保留**（环境回收后仍在）。
> 把本 notebook 与 `ml/` 整个文件夹上传到 `work/` 下最省事。

> **数据量提示（重要）**：当前自采样本仅有 `ml/raw_mine/labels/` 下的少量 bbox 标签
> （如 3 张 undercut）。seg 训练是 bootstrap，精度受样本量限制；要更高精度，
> 往 `ml/raw_mine/labels/` + `ml/raw_mine/images/` 补更多带标签图后重跑本 notebook 即可。
> 类别顺序与 `yolov8n_seg_weld.yaml` 的 `names` 必须一致（0=porosity…4=unfused）。

In [ ]:
# ===== 配置区（默认已指向持久目录 work/）=====
import os
WORK = '/home/aistudio/work'
ML   = WORK + '/ml'
SRC_LABELS = ML + '/raw_mine/labels'          # 源：YOLO 检测标签（每行 class cx cy w h）
SRC_IMAGES = ML + '/raw_mine/images'          # 源：原图（自动按标签 stem 配对，过滤 _annotated/_crop）
DST = ML + '/dataset_weld'                     # 组装后的数据集根
IMG_TRAIN = DST + '/images/train'
IMG_VAL   = DST + '/images/val'
LBL_TRAIN = DST + '/labels/train'              # ⚠️ 必须叫 labels（Ultralytics 把 images->labels 配对）
LBL_VAL   = DST + '/labels/val'
YAML = ML + '/yolov8n_seg_weld.yaml'
SPLIT_VAL = 0.2   # 验证集比例（样本极少时至少留 1 张）
print('源标签:', SRC_LABELS)
print('源图片:', SRC_IMAGES)
print('输出根:', DST)

In [ ]:
import os

def bbox_to_polygon(cx, cy, w, h):
    x1 = max(0.0, min(1.0, cx - w / 2.0))
    y1 = max(0.0, min(1.0, cy - h / 2.0))
    x2 = max(0.0, min(1.0, cx + w / 2.0))
    y2 = max(0.0, min(1.0, cy + h / 2.0))
    return [x1, y1, x2, y1, x2, y2, x1, y2]

def convert_file(src_path, dst_path):
    with open(src_path, 'r', encoding='utf-8') as f:
        lines = [ln.strip() for ln in f if ln.strip()]
    out = []
    for ln in lines:
        parts = ln.split()
        if len(parts) < 5:
            continue
        cls, cx, cy, w, h = parts[0], float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        poly = bbox_to_polygon(cx, cy, w, h)
        out.append(' '.join([str(cls)] + [f'{p:.6f}' for p in poly]))
    os.makedirs(os.path.dirname(dst_path), exist_ok=True)
    with open(dst_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(out) + ('\n' if out else ''))
    return len(out)

In [ ]:
# ===== 组装数据集：配对图片 + bbox->seg 转换 + 切分 train/val =====
import shutil, random
random.seed(42)

# 清理旧输出，保证幂等
for d in (IMG_TRAIN, IMG_VAL, LBL_TRAIN, LBL_VAL):
    if os.path.isdir(d):
        shutil.rmtree(d)

# 收集「有对应图片」的标签 stem（自动过滤 _annotated/_crop 等无标签图）
stems = []
for fn in sorted(os.listdir(SRC_LABELS)):
    if not fn.endswith('.txt'):
        continue
    stem = fn[:-4]
    imgp = None
    for ext in ('.jpg', '.jpeg', '.png', '.bmp', '.JPG', '.PNG'):
        cand = os.path.join(SRC_IMAGES, stem + ext)
        if os.path.isfile(cand):
            imgp = cand
            break
    if imgp:
        stems.append((stem, imgp))
    else:
        print('⚠️ 跳过', stem, '：在', SRC_IMAGES, '找不到对应图片')

print('可用样本:', len(stems))
if not stems:
    raise SystemExit('没有可用样本，请检查 SRC_LABELS / SRC_IMAGES 路径与文件')

# 切分（样本极少时 val 至少 1 张）
n_val = max(1, int(round(len(stems) * SPLIT_VAL))) if len(stems) > 1 else 0
random.shuffle(stems)
val_stems = stems[:n_val]
tr_stems  = stems[n_val:]

def place(pair_list, img_dir, lbl_dir):
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)
    cnt = 0
    for stem, imgp in pair_list:
        shutil.copy(imgp, os.path.join(img_dir, os.path.basename(imgp)))
        convert_file(os.path.join(SRC_LABELS, stem + '.txt'),
                     os.path.join(lbl_dir, stem + '.txt'))
        cnt += 1
    return cnt

ntr = place(tr_stems, IMG_TRAIN, LBL_TRAIN)
nva = place(val_stems, IMG_VAL, LBL_VAL)
print(f'已组装：train {ntr} 张图, val {nva} 张图 -> {DST}')

In [ ]:
# ===== 校验 =====
def count(d):
    return len([f for f in os.listdir(d) if os.path.isfile(os.path.join(d, f))]) if os.path.isdir(d) else 0

print('images/train:', count(IMG_TRAIN), '| images/val:', count(IMG_VAL))
print('labels/train:', count(LBL_TRAIN), '| labels/val:', count(LBL_VAL))

sample_dir = LBL_TRAIN if count(LBL_TRAIN) else LBL_VAL
fs = [f for f in os.listdir(sample_dir) if f.endswith('.txt')]
if fs:
    with open(os.path.join(sample_dir, fs[0]), 'r', encoding='utf-8') as f:
        print('示例 seg 标签', fs[0], ':')
        for ln in f.read().strip().splitlines()[:2]:
            print('  ', ln)

## 下一步：在 AI Studio 直接训练 yolov8-seg

下方训练 cell 已带**断点续连**：每个 epoch 末自动把 `last.pt` 备份到
`/home/aistudio/work/weld_seg_last_backup.pt`；AI Studio 免费环境回收/杀进程后，
重新运行该 cell 即自动 `resume` 续训，无需从头跑（解决之前 120 轮被踢重跑的痛点）。

训练完成后导出 **`WeldSegModel.mlpackage`**（独立的 seg 模型，**不覆盖**已有的
bbox 检测模型 `WeldDefectModel.mlpackage`），供 App 端 MetricSizer 使用。

In [ ]:
# ===== 训练 yolov8-seg（B v2，带断点续连 + 每 epoch 备份）=====
# AI Studio 免费环境会回收/杀进程：本 cell 每个 epoch 末自动备份 last.pt；
# 被踢后重新运行即自动 resume 续训，直到 TOTAL_EPOCHS。
import os, glob as _glob, shutil as _shutil
from ultralytics import YOLO

SEG_NAME = 'weld_seg_v1'
TOTAL_EPOCHS = 120
BACKUP_PT = '/home/aistudio/work/weld_seg_last_backup.pt'
ML = '/home/aistudio/work/ml'

# 直接传 dict 数据集，path 用绝对路径，避免 yaml 内 path:. 相对 CWD 解析歧义
# （AI Studio 回收后 CWD 常重置为 /home/aistudio，相对解析会找不到 dataset_weld）
DATA = {
    'path': ML,
    'train': 'dataset_weld/images/train',
    'val':   'dataset_weld/images/val',
    'nc': 5,
    'names': ['porosity', 'crack', 'undercut', 'overlap', 'unfused'],
}

!pip install ultralytics -q

def _find_last():
    cands = sorted(_glob.glob(f'runs/segment/{SEG_NAME}*/weights/last.pt'))
    return cands[-1] if cands else None

def _on_epoch_end(trainer):
    lp = _find_last()
    if lp and os.path.isfile(lp):
        try:
            _shutil.copy(lp, BACKUP_PT)
            ep = getattr(trainer, 'epoch', None)
            if ep is not None and ep % 10 == 0:
                print(f'[备份] epoch {ep} 末已备份 last.pt -> {BACKUP_PT}')
        except Exception:
            pass

last_pt = _find_last()
if last_pt:
    print('[续训] 检测到', last_pt, '-> 从中断处继续到', TOTAL_EPOCHS, '轮')
    model = YOLO(last_pt)
    try:
        model.add_callback('on_train_epoch_end', _on_epoch_end)
    except Exception:
        pass
    model.train(resume=True)
else:
    print('[新训] 未找到历史权重，从 yolov8n-seg.pt 开始')
    model = YOLO('yolov8n-seg.pt')
    try:
        model.add_callback('on_train_epoch_end', _on_epoch_end)
    except Exception:
        pass
    model.train(data=DATA, task='segment', epochs=TOTAL_EPOCHS, imgsz=640, batch=16,
                name=SEG_NAME, patience=30, augment=True,
                hsv_h=0.015, hsv_s=0.9, hsv_v=0.5,
                fliplr=0.5, mosaic=1.0, mixup=0.1,
                degrees=5.0, scale=0.5, shear=1.0, perspective=0.0005,
                copy_paste=0.45, seed=42)
print('训练完成')

# 导出独立的 seg 模型（INT8 权重量化，跑 Neural Engine）—— 不覆盖 bbox 模型
exported = model.export(format='coreml', nms=True, quantize='w8a16', imgsz=640)
pkg_name = 'WeldSegModel.mlpackage' if os.path.isdir(exported) else 'WeldSegModel.mlmodel'
print('导出产物:', exported)
print('-> 重命名为', pkg_name, '后加入 Xcode 工程 app_ios/native/WeldFatigueChecker/（与 WeldDefectModel.mlpackage 并列，不覆盖）')

## 断点续连 / 环境回收恢复 SOP

1. **运行中**：每个 epoch 末自动备份 `last.pt` -> `/home/aistudio/work/weld_seg_last_backup.pt`。
2. **被踢（进程被杀，work/ 仍在）**：直接重新运行上方训练 cell，脚本自动检测 `last.pt`
   并 `resume=True` 续训，epoch 累加直到 120。
3. **work/ 被清（极端）**：若你此前用侧边「下载文件」存过 `weld_seg_last_backup.pt`——
   重新上传该文件为 `runs/segment/weld_seg_v1/weights/last.pt`（首次需先跑一次新训建目录，
   或先 `YOLO('yolov8n-seg.pt').train(...)` 生成目录再覆盖 last.pt），重跑训练 cell 即无缝续训。
4. **从未训过**：直接运行训练 cell 即从头训（仍带备份，安全）。